# Projet Final : Biais et équité dans la classification d'images médicales
## NIH Chest X-ray

**Groupe 3 :** Othmane Nammous, Tharushan Uthayakumar, Rémi Malapert - CPES3

**Date de rendu :** 3 avril 2026

# 0/- Introduction : contexte, enjeux et objectifs du projet final

## 1) Contexte et cas d'usage

### 1.1 Diagnostic médical assisté par IA
Les radiographies thoraciques (chest X-rays) font partie des examens d'imagerie les plus utilisés en pratique clinique. Des modèles de deep learning comme ResNet18 peuvent aider au tri diagnostique en prédisant la présence d'une pathologie à partir de l'image.

Cependant, la performance prédictive ne suffit pas à garantir un usage fiable en santé. Un modèle peut être globalement performant tout en restant moins fiable pour certains sous-groupes de patients. L'enjeu du projet est donc d'évaluer à la fois la performance du modèle et son équité algorithmique.

### 1.2 Données étudiées
Ce projet s'appuie sur un sous-ensemble du dataset NIH Chest X-ray 14 (Wang et al., 2017), organisé sous forme de dossiers train et valid avec des images classées en malade et sain, ainsi qu'un fichier metadata.csv contenant les informations patient et examen.

Les variables étudiées sont principalement les métadonnées structurées: âge, genre, position de vue et labels cliniques. Le projet final se concentre sur une tâche binaire malade/sain, puis sur l'analyse des biais associés à deux attributs sensibles:
- le genre du patient,
- la position de vue (PA/AP).

## 2) Objectif et méthodologie
Ce notebook vise à identifier les configurations offrant le meilleur compromis entre utilité clinique (performance) et réduction des biais algorithmiques (fairness) sur un pipeline de classification d'images médicales.

Pour évaluer ce compromis, nos analyses s'appuieront systématiquement sur deux familles de métriques :

La performance : mesurée par la Balanced Accuracy.

La fairness : évaluée à l'aide de la Statistical Parity Difference (SPD), du Disparate Impact Ratio (DIR), de l'Equal Opportunity Difference (EOD) et de l'Average Odds Difference (AOD).

Pour mener à bien cette évaluation, notre méthodologie s'articule en trois grandes étapes :

1. Analyse rapide des données : Étude concise du dataset afin de mettre en évidence les principaux déséquilibres. Nous analyserons la distribution des labels, du genre, de la position de vue (PA/AP) et de l'âge, et calculerons les métriques de fairness initiales pour repérer les biais historiques.

2. Pré-processing (Impact de la pondération) : Comparaison de plusieurs méthodes d'entraînement en amont. Nous évaluerons une baseline sans pondération, puis la comparerons avec différentes approches de Reweighing (par genre, par position, et croisé genre × position) afin d'observer l'effet direct de la pondération sur les performances et les écarts entre groupes.

3. Post-processing et Combinaisons : Test d'approches correctives appliquées en aval des prédictions. Nous utiliserons les méthodes Reject Option Classification et Calibrated Equalized Odds, que nous appliquerons à la fois sur la baseline et sur les modèles pondérés. Cette étude des combinaisons (pre-processing + post-processing) nous permettra d'isoler l'approche la plus robuste selon l'attribut sensible ciblé.

# Chargement des données et préparation


In [69]:
# === Imports ===
import os
import random
import shutil
import warnings
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.metrics import accuracy_score, balanced_accuracy_score
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder
import torch

warnings.simplefilter(action="ignore", category=FutureWarning)
warnings.simplefilter(action="ignore", append=True, category=UserWarning)

# Métriques de fairness (AIF360 sklearn API)
from aif360.sklearn.metrics import (
    statistical_parity_difference,
    disparate_impact_ratio,
    equal_opportunity_difference,
    average_odds_difference,
    base_rate,
    smoothed_edf,
    df_bias_amplification,
    conditional_demographic_disparity,
)
#from sklearn.metrics import balanced_accuracy_score, accuracy_score

# AIF360 pour le post-processing
from aif360.datasets import BinaryLabelDataset
from aif360.metrics import ClassificationMetric
from aif360.algorithms.postprocessing.reject_option_classification import RejectOptionClassification
from aif360.algorithms.postprocessing.calibrated_eq_odds_postprocessing import CalibratedEqOddsPostprocessing

# Entraînement du modèle
#from train_classifieur import train_classifier, pred_classifier
#import torch
from train_classifieur import (
    ChestXRayClassifier,
    pred_classifier,
    preds_todf,
    train_classifier,
    transforms_valid,
)

In [70]:
import os
print(os.getcwd())

c:\Users\remim\Documents\Projet Fairness\Fairness-in-AI


In [71]:
# === Configuration ===
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device utilisé : {device}")
if device.type == 'cuda':
    print(f"GPU : {torch.cuda.get_device_name(0)}")
    print(f"Mémoire allouée : {round(torch.cuda.memory_allocated(0)/1024**3,1)} GB")

# Chemins portables pour le workspace courant
BASE_DIR = os.getcwd()
DATA_DIR = os.path.join(BASE_DIR, "Data")

# Sorties d'expériences hors dépôt (temp système) pour éviter de recréer expe_log
EXPE_DIR = os.path.join(os.environ.get("TEMP", BASE_DIR), "fairness_runs")
WORK_DIR = EXPE_DIR
CSV_PATH = os.path.join(DATA_DIR, "metadata.csv")

os.makedirs(EXPE_DIR, exist_ok=True)
print(f"Dossier des sorties: {EXPE_DIR}")

SMOKE_TEST = True
MAX_EPOCHS = 1 if SMOKE_TEST else 25
print(f"Mode smoke test: {SMOKE_TEST} | max_epochs={MAX_EPOCHS}")

Device utilisé : cpu
Dossier des sorties: C:\Users\remim\AppData\Local\Temp\fairness_runs
Mode smoke test: True | max_epochs=1


## I/- Analyse exploratoire des données

Cette première partie joue le rôle de mini mi-projet : on commence par décrire le dataset, on nettoie les valeurs manifestement impossibles, puis on identifie les déséquilibres qui peuvent influencer l'apprentissage futur d'un modèle.

L'analyse se concentre sur les variables qui portent un risque d'effet de groupe : l'âge, le genre, la position de vue et la variable cible malade / sain.

### 1.1 Préparation et nettoyage des données

Pour l'analyse de fairness, il faut une cible binaire claire. Nous créons donc la colonne `label` puis la variable binaire `label_binary` pour distinguer les patients sains des patients malades.

L'attribut sensible principal est le **genre du patient**. Nous le codons en binaire :
- **1** pour les hommes (M), groupe de référence car majoritaire dans le dataset,
- **0** pour les femmes (F), groupe comparatif.

Ce choix ne signifie pas que le groupe de référence est réellement favorisé. Il sert uniquement à calculer des métriques de parité de manière standardisée.

Nous analysons aussi la **position de vue** (AP / PA), car c'est un attribut technique susceptible de devenir un proxy du diagnostic. Enfin, l'âge est gardé comme variable numérique de contexte : il permet de vérifier si la distribution des patients est homogène ou si certaines classes d'âge sont sous-représentées.

L'objectif de cette partie est de construire une baseline interprétable avant toute mitigation : on comprend d'abord les données, on mesure ensuite les déséquilibres, puis seulement on corrige.

In [72]:
def add_binary_columns(df):
    df_out = df.copy()
    df_out['label_binary'] = (df_out['label'] == 'malade').astype(int)
    df_out['Gender_Binary'] = (df_out['Patient Gender'] == 'M').astype(int)
    df_out['Position_Binary'] = (df_out['View Position'] == 'PA').astype(int)
    return df_out


def prepare_experiment_data():
    """Charge le CSV existant et conserve le split train/valid déjà présent."""
    df_split = add_binary_columns(pd.read_csv(CSV_PATH))

    if 'train_valid' not in df_split.columns:
        raise ValueError(
            "La colonne 'train_valid' est absente de metadata.csv. "
            "Le mode simple attend un split train/valid déjà défini."
        )

    # Normalisation des libellés de split pour éviter les ambiguïtés
    df_split['train_valid'] = (
        df_split['train_valid']
        .astype(str)
        .str.strip()
        .str.lower()
        .replace({'val': 'valid', 'validation': 'valid'})
    )

    allowed = {'train', 'valid'}
    unknown = sorted(set(df_split['train_valid'].unique()) - allowed)
    if unknown:
        raise ValueError(f"Valeurs de split non supportées en mode simple: {unknown}")

    df_split['train_valid'] = pd.Categorical(
        df_split['train_valid'],
        categories=['train', 'valid'],
        ordered=True,
    )

    df_split.to_csv(CSV_PATH, index=False)
    return df_split.copy(), df_split


def predict_partition(csv_path, split_root, partition, ckpt_path, preds_col='preds'):
    """Prédit une partition ImageFolder (train/valid) et prépare les colonnes utiles à l'évaluation."""
    train_dataset = ImageFolder(os.path.join(split_root, 'train'), transform=transforms_valid)
    label_decoder = {v: k for k, v in train_dataset.class_to_idx.items()}
    dataset = ImageFolder(os.path.join(split_root, partition), transform=transforms_valid)

    model = ChestXRayClassifier(adamax=True, cosine=True, nb_classes=len(label_decoder))
    model.load_state_dict(torch.load(ckpt_path, map_location='cpu')['state_dict'])
    model.eval()

    df_partition = pd.read_csv(csv_path)
    df_partition = df_partition[df_partition['train_valid'] == partition].copy()
    df_partition[preds_col] = None
    df_partition = preds_todf(df_partition, dataset, label_decoder, model, preds_col)

    # Calcul de score_malade (probabilité de la classe 0 = malade) depuis les logits
    logit0 = df_partition[f'{preds_col}_logit0'].values
    logit1 = df_partition[f'{preds_col}_logit1'].values
    logits = np.column_stack([logit0, logit1])
    exp_logits = np.exp(logits - logits.max(axis=1, keepdims=True))  # Stabilité numérique
    probs = exp_logits / exp_logits.sum(axis=1, keepdims=True)
    df_partition['score_malade'] = probs[:, 0]

    # Colonnes binaires standardisées pour les cellules d'évaluation
    truth_col = 'labels' if 'labels' in df_partition.columns else 'label'
    df_partition['label_binary'] = (df_partition[truth_col] == 'malade').astype(int)
    df_partition['preds_binary'] = (df_partition[preds_col] == 'malade').astype(int)
    if 'Gender_Binary' not in df_partition.columns:
        df_partition['Gender_Binary'] = (df_partition['Patient Gender'] == 'M').astype(int)
    if 'Position_Binary' not in df_partition.columns:
        df_partition['Position_Binary'] = (df_partition['View Position'] == 'PA').astype(int)

    return df_partition


df_raw, df_split = prepare_experiment_data()
df = df_split
print(f"Split train/valid prêt: {df_split['train_valid'].value_counts().to_dict()}")
print(f"CSV utilisé: {CSV_PATH}")

Split train/valid prêt: {'train': 3976, 'valid': 1246}
CSV utilisé: c:\Users\remim\Documents\Projet Fairness\Fairness-in-AI\Data\metadata.csv


In [73]:
df.describe()


,Follow-up #,Patient ID,Patient Age,OriginalImage[Width,Height],OriginalImagePixelSpacing[x,y],Unnamed: 11,WEIGHTS,label_binary,Gender_Binary,Position_Binary
count,5222.000000,5222.000000,5222.000000,5222.000000,5222.000000,5222.000000,5222.000000,0.0,5222.0,5222.000000,5222.000000,5222.000000
mean,8.023746,14087.229223,44.992149,2639.062428,2489.886059,0.155783,0.155783,NaN,1.0,0.452700,0.595557,0.612792
std,12.636620,8710.367565,18.289353,353.373602,394.052927,0.016832,0.016832,NaN,0.0,0.497805,0.490831,0.487158
min,0.000000,8.000000,2.000000,1490.000000,1297.000000,0.139000,0.139000,NaN,1.0,0.000000,0.000000,0.000000
25%,0.000000,6674.000000,30.000000,2500.000000,2048.000000,0.143000,0.143000,NaN,1.0,0.000000,0.000000,0.000000
50%,3.000000,13557.000000,46.000000,2530.000000,2544.000000,0.143000,0.143000,NaN,1.0,0.000000,1.000000,1.000000
75%,10.000000,20725.000000,58.000000,2992.000000,2991.000000,0.168000,0.168000,NaN,1.0,1.000000,1.000000,1.000000
max,76.000000,30795.000000,412.000000,3056.000000,3056.000000,0.194314,0.194314,NaN,1.0,1.000000,1.000000,1.000000


In [74]:
missing_values = df.isnull().sum()
print(missing_values)

Image Index                       0
Finding Labels                    0
Follow-up #                       0
Patient ID                        0
Patient Age                       0
Patient Gender                    0
View Position                     0
OriginalImage[Width               0
Height]                           0
OriginalImagePixelSpacing[x       0
y]                                0
Unnamed: 11                    5222
train_valid                       0
label                             0
WEIGHTS                           0
label_binary                      0
Gender_Binary                     0
Position_Binary                   0
dtype: int64


En observant le `describe()`, on remarque que l'âge maximum est de **412 ans**, ce qui est impossible. Ce type de valeur aberrante fausse à la fois les statistiques descriptives et les analyses de fairness, en particulier dès qu'on compare les groupes d'âge.

Nous conservons donc uniquement les patients dont l'âge est inférieur à 100 ans. 

Par ailleurs, nous supprimerons la colonne `Unnamed: 11` qui est vide.

In [75]:
# === Nettoyage et aperçu des données ===
df = df_raw.copy()

df.drop(columns=['Unnamed: 11'], inplace=True, errors='ignore')

# Filtrage des âges aberrants dès la préparation
df_before_age_filter = df.copy()
df = df[df['Patient Age'] < 100].copy()
n_removed_age = len(df_before_age_filter) - len(df)

split_counts = df_split['train_valid'].value_counts().reindex(['train', 'valid']).fillna(0).astype(int)
label_counts = df['label'].value_counts().reindex(['sain', 'malade']).fillna(0).astype(int)
gender_counts = df['Patient Gender'].value_counts().reindex(['F', 'M']).fillna(0).astype(int)
position_counts = df['View Position'].value_counts().reindex(['AP', 'PA']).fillna(0).astype(int)

# Palette proche des defaults Plotly utilises dans le mi-projet
plotly_colors = ['#636EFA', '#EF553B', '#00CC96', '#AB63FA']

def add_count_bar(fig, series, row, col, color):
    total = series.sum()
    df_plot = pd.DataFrame({'cat': series.index.astype(str), 'count': series.values})
    df_plot['pct'] = 100 * df_plot['count'] / total if total > 0 else 0.0
    df_plot['text'] = df_plot['count'].astype(str) + ' (' + df_plot['pct'].round(1).astype(str) + '%)'
    fig.add_trace(
        go.Bar(
            x=df_plot['cat'],
            y=df_plot['count'],
            marker_color=color,
            text=df_plot['text'],
            textposition='outside',
            cliponaxis=False,
            showlegend=False,
            hovertemplate='%{x}: %{y}<extra></extra>',
        ),
        row=row,
        col=col,
    )

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        'Répartition train / valid', 'Labels (sain vs malade)',
        'Genre', 'Position de vue'
    ],
    specs=[[{'type': 'bar'}, {'type': 'bar'}],
           [{'type': 'bar'}, {'type': 'bar'}]]
)

add_count_bar(fig, split_counts, 1, 1, plotly_colors[0])
add_count_bar(fig, label_counts, 1, 2, plotly_colors[1])
add_count_bar(fig, gender_counts, 2, 1, plotly_colors[2])
add_count_bar(fig, position_counts, 2, 2, plotly_colors[3])

fig.update_xaxes(title_text='')
fig.update_yaxes(title_text="Nombre d'images")
fig.update_layout(
    height=760,
    template='plotly',
    title_text='Aperçu du dataset après nettoyage',
    margin=dict(l=40, r=40, t=90, b=40),
)
fig.show()





**Analyse graphique des 4 distributions**

1. **Répartition train / valid**
- On observe un split d'environ **76% train / 24% valid**.
- Cette proportion est cohérente pour entraîner un modèle tout en gardant un échantillon de validation suffisamment grand.

2. **Labels (sain vs malade)**
- La distribution est **modérément équilibrée**: ~55% sain vs ~45% malade.
- Ce déséquilibre reste raisonnable et limite le risque d'un modèle trivial qui prédit majoritairement une seule classe.
- Cela justifie l'usage de la **Balanced Accuracy** dans la suite, plus adaptée qu'une accuracy classique.

3. **Genre**
- Les hommes sont majoritaires (~60%) par rapport aux femmes (~40%).
- Ce déséquilibre peut introduire un risque de performance inégale entre groupes de genre.
- Il faudra donc vérifier les métriques de fairness (SPD, DIR, EOD, AOD) spécifiquement sur cet attribut.

4. **Position de vue (AP/PA)**
- La position **PA est majoritaire** (~61%) devant AP (~39%).
- La position de vue étant liée au contexte d'acquisition, ce déséquilibre peut agir comme un **biais technique** dans la prédiction.
- Cet écart confirme l'intérêt d'analyser explicitement la fairness sur l'attribut Position dans les parties suivantes.

**Synthèse rapide**
- La base est exploitable, mais les déséquilibres sur **Genre** et **Position** justifient les étapes de mitigation testées par la suite (reweighing puis post-processing).

In [76]:
age_fig = px.histogram(
    df,
    x='Patient Age',
    color='Patient Gender',
    nbins=30,
    opacity=0.7,
    marginal='box',
    barmode='overlay',
    title="Distribution de l'âge après filtrage des valeurs aberrantes",
    labels={'Patient Age': 'Âge', 'Patient Gender': 'Genre'},
)
age_fig.update_layout(template='plotly', height=500)
age_fig.show()

L'âge est globalement centré autour de la cinquantaine pour les deux genres. La forme des distributions est proche d'une cloche, avec une baisse nette des effectifs après 60 ans. Cette baisse ne traduit pas forcément un biais de collecte, car cela peut simplement montrer une structure démographique réaliste, avec moins d'individus très âgés dans l'échantillon.

Le point important ici n'est pas la moyenne, mais les extrémités de distribution. Les patients de plus de 80 ans et de moins de 10 ans sont peu nombreux : si le modèle est ensuite entraîné sans précaution, il risque d'être moins robuste pour cette tranche d'âge et de produire des erreurs systématiques sur ce sous-groupe.

On peut donc retenir deux idées :
- l'âge ne semble pas créer de rupture brutale dans la distribution globale,
- mais il reste un facteur de sous-représentation potentiel, donc un risque de biais de performance au moment de l'apprentissage.

In [77]:
summary_df = pd.DataFrame({
    'Mesure': [
        'Images avant filtrage',
        'Images après filtrage',
        'Images retirées',
        'Taux de maladie',
        "Proportion d'hommes",
        'Proportion de PA',
        'Âge moyen',
        'Âge médian',
    ],
    'Valeur': [
        len(df_before_age_filter),
        len(df),
        n_removed_age,
        f"{(df['label'] == 'malade').mean():.2%}",
        f"{(df['Patient Gender'] == 'M').mean():.2%}",
        f"{(df['View Position'] == 'PA').mean():.2%}",
        f"{df['Patient Age'].mean():.1f}",
        f"{df['Patient Age'].median():.1f}",
    ]
})
summary_df

,Mesure,Valeur
0,Images avant filtrage,5222
1,Images après filtrage,5219
2,Images retirées,3
3,Taux de maladie,45.28%
4,Proportion d'hommes,59.55%
5,Proportion de PA,61.26%
6,Âge moyen,44.9
7,Âge médian,46.0


### 1.2 Analyse descriptive et biais initiaux

In [78]:
# === Analyse descriptive visuelle (sans graphique d'âge) ===

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        'Label par genre', 'Label par position',
        'Prévalence par sous-groupe', ''
    ],
    specs=[[{'type': 'bar'}, {'type': 'bar'}],
           [{'type': 'bar'}, None]]
)

ct_gender = (
    pd.crosstab(df['Patient Gender'], df['label'], normalize='index') * 100
).reindex(index=['F', 'M'], columns=['malade', 'sain'], fill_value=0)

fig.add_trace(
    go.Bar(
        x=['F', 'M'],
        y=ct_gender['malade'].tolist(),
        text=ct_gender['malade'].round(1).astype(str) + '%',
        textposition='outside',
        name='Malade',
        marker_color='#EF553B',
    ),
    row=1,
    col=1,
)
fig.add_trace(
    go.Bar(
        x=['F', 'M'],
        y=ct_gender['sain'].tolist(),
        text=ct_gender['sain'].round(1).astype(str) + '%',
        textposition='outside',
        name='Sain',
        marker_color='#636EFA',
    ),
    row=1,
    col=1,
)

ct_pos = (
    pd.crosstab(df['View Position'], df['label'], normalize='index') * 100
).reindex(index=['AP', 'PA'], columns=['malade', 'sain'], fill_value=0)

fig.add_trace(
    go.Bar(
        x=['AP', 'PA'],
        y=ct_pos['malade'].tolist(),
        text=ct_pos['malade'].round(1).astype(str) + '%',
        textposition='outside',
        name='Malade',
        showlegend=False,
        marker_color='#EF553B',
    ),
    row=1,
    col=2,
)
fig.add_trace(
    go.Bar(
        x=['AP', 'PA'],
        y=ct_pos['sain'].tolist(),
        text=ct_pos['sain'].round(1).astype(str) + '%',
        textposition='outside',
        name='Sain',
        showlegend=False,
        marker_color='#636EFA',
    ),
    row=1,
    col=2,
)

ct_cross = df.groupby(['Patient Gender', 'View Position'])['label_binary'].mean().reset_index()
ct_cross['group'] = ct_cross['Patient Gender'] + ' / ' + ct_cross['View Position']
ct_cross['prevalence_pct'] = (100 * ct_cross['label_binary']).round(1)
fig.add_trace(
    go.Bar(
        x=ct_cross['group'].tolist(),
        y=ct_cross['label_binary'].tolist(),
        text=ct_cross['prevalence_pct'].astype(str) + '%',
        textposition='outside',
        showlegend=False,
        marker_color='#00CC96',
    ),
    row=2,
    col=1,
)

fig.update_yaxes(title_text='Pourcentage (%)', range=[0, 100], row=1, col=1)
fig.update_yaxes(title_text='Pourcentage (%)', range=[0, 100], row=1, col=2)
fig.update_yaxes(title_text='Prévalence', tickformat='.0%', row=2, col=1)
fig.update_layout(
    barmode='group',
    height=780,
    template='plotly',
    title_text='Lecture rapide des déséquilibres structurants',
    title_x=0.05,
    legend_title_text='Lecture des classes',
)
fig.show()

stats = pd.DataFrame({
    'Indicateur': ['Taux de maladie', 'Hommes', 'PA', 'Âge moyen', 'Âge médian'],
    'Valeur': [
        f"{(df['label'] == 'malade').mean():.2%}",
        f"{(df['Patient Gender'] == 'M').mean():.2%}",
        f"{(df['View Position'] == 'PA').mean():.2%}",
        f"{df['Patient Age'].mean():.1f}",
        f"{df['Patient Age'].median():.1f}",
    ]
})
stats

,Indicateur,Valeur
0,Taux de maladie,45.28%
1,Hommes,59.55%
2,PA,61.26%
3,Âge moyen,44.9
4,Âge médian,46.0


Grace à ces graphiques on peut visualiser plusieurs choses

- Premièrement, la variable cible est assez équilibrée globalement entre hommes et femmes : on observe seulement un léger écart de taux de maladie. Cela veut dire que le dataset ne contient pas de biais massif au niveau agrégé, mais cette lecture globale peut masquer des différences plus fines.

- Deuxièmement, la position de vue est plus problématique. La répartition AP / PA n'est pas neutre : la vue PA présente une proportion de malades plus faible que la vue AP. Cet écart suggère que la position de vue peut servir de proxy technique pour le diagnostic, ce qui constitue un risque classique de biais de collecte.

- La comparaison croisée genre x position montre en revanche que les deux genres sont répartis de manière assez proche entre AP et PA. Autrement dit, la position de vue n'explique pas à elle seule les écarts observés entre hommes et femmes : elle est un biais propre, mais pas un facteur confondant majeur entre genres.

### 1.3 Outils de mesure (performance et fairness)

Dans cette partie nous calculons les métriques de fairness directement sur les données brutes. Comme il n'y a pas encore de prédictions de modèle, cela permet d'isoler les biais présents dans la distribution elle-même.

Les trois indicateurs retenus sont les suivants :

- **Base rate** : proportion de la classe positive, ici la part d'images pathologiques
- **Statistical Parity Difference (SPD)** : différence de taux de positifs entre le groupe non privilégié et le groupe de référence
- **Disparate Impact Ratio (DIR)** : ratio des taux de positifs entre les deux groupes

Pour rappel :
- un SPD proche de 0 si le dataset est équilibré,
- un DIR proche de 1 si le taux de maladie ne varie pas fortement selon le groupe étudié.

L'intérêt de ce cadre est ainsi d'obtenir une baseline interprétable avant la mitigation. Si le dataset est déjà biaisé à ce stade, le modèle risque de le reproduire. Si la baseline est faible, une correction devra être justifiée plus tard par la partie mitigation.

In [79]:
# === Métriques de fairness sur les données brutes (labels réels) ===

spd_g = statistical_parity_difference(
    y_true=df['label_binary'], prot_attr=df['Gender_Binary'], pos_label=1)
dir_g = disparate_impact_ratio(
    y_true=df['label_binary'], prot_attr=df['Gender_Binary'], pos_label=1)

spd_p = statistical_parity_difference(
    y_true=df['label_binary'], prot_attr=df['Position_Binary'], pos_label=1)
dir_p = disparate_impact_ratio(
    y_true=df['label_binary'], prot_attr=df['Position_Binary'], pos_label=1)

fairness_summary = pd.DataFrame({
    'Attribut': ['Genre', 'Position'],
    'SPD': [spd_g, spd_p],
    'DIR': [dir_g, dir_p],
})


display(
    fairness_summary.style
    .format({'SPD': '{:+.4f}', 'DIR': '{:.4f}'})
    .hide(axis='index')
    .set_properties(**{'text-align': 'center'})
    .set_table_styles([
        {'selector': 'th', 'props': [('text-align', 'center')]},
        {'selector': 'caption', 'props': [('caption-side', 'top'), ('font-weight', 'bold')]},
    ])
    .set_caption('Métriques de fairness sur les labels bruts')
)

Attribut,SPD,DIR
Genre,-0.0364,0.9221
Position,+0.0908,1.2175


Statistical Parity Difference (SPD) :
- genre = -0.0364 
- position = +0.0908

Un SPD aussi proche de 0 indique une quasi-parité statistique entre les genres, et entre les positions sur la variable agrégée has_pathology.

Disparate Impact Ratio (DIR) :
- genre = 0,991 : Ce ratio est très proche de la parité parfaite (1.0) et se situe largement dans la zone de tolérance des 80% (0.8 - 1.25). Il n'y a donc pas d'impact disproportionné majeur lié au genre sur la prédiction globale.

- Position = 1.217 : Le ratio est à la limite de la fourchette d'acceptabilité (règle des 80% plafonnée à 1.25), et indique un certain déséquilibre existant. Cela signifie que l'une des positions (AP) a environ 21% de probabilité supplémentaire d'être classifiée positive par rapport à la vue standard (PA). Dans un contexte clinique, ce biais technique est significatif et justifie l'application de méthodes de mitigation.


Pour aller au-delà de cette première lecture descriptive, nous avons maintenant besoin d’outils plus généraux pour standardiser l’évaluation de nos expérimentations. 

La fonction présentée ci-dessous centralise le calcul des métriques de performance et de fairness, afin de pouvoir comparer proprement la baseline, les modèles pondérés et les différentes méthodes de post-processing. 

Les fonctions suivantes servent ensuite à construire les poids de reweighing et à préparer les prédictions, ce qui permettra d’évaluer chaque stratégie de mitigation sur une base commune.

In [80]:
def get_metrics(y_true, y_pred=None, prot_attr=None, priv_group=1, pos_label=1, sample_weight=None):
    """
    Calcule un ensemble complet de métriques de fairness et de performance.
    Adapté du TD3/TD4 (aif360 sklearn API).

    Args:
        y_true:   array-like de vérité terrain (0/1)
        y_pred:   array-like de prédictions (0/1)
        prot_attr: array-like de l'attribut sensible
        priv_group: valeur du groupe privilégié
        pos_label: valeur du label positif
        sample_weight: poids des échantillons (optionnel)

    Returns:
        dict de métriques
    """
    group_metrics = {}
    group_metrics["base_rate_truth"] = base_rate(
        y_true=y_true, pos_label=pos_label, sample_weight=sample_weight
    )
    group_metrics["statistical_parity_difference"] = statistical_parity_difference(
        y_true=y_true, y_pred=y_pred, prot_attr=prot_attr, priv_group=priv_group,
        pos_label=pos_label, sample_weight=sample_weight
    )
    group_metrics["disparate_impact_ratio"] = disparate_impact_ratio(
        y_true=y_true, y_pred=y_pred, prot_attr=prot_attr, priv_group=priv_group,
        pos_label=pos_label, sample_weight=sample_weight
    )
    if y_pred is not None:
        group_metrics["base_rate_preds"] = base_rate(
            y_true=y_pred, pos_label=pos_label, sample_weight=sample_weight
        )
        group_metrics["equal_opportunity_difference"] = equal_opportunity_difference(
            y_true=y_true, y_pred=y_pred, prot_attr=prot_attr, priv_group=priv_group,
            pos_label=pos_label, sample_weight=sample_weight
        )
        group_metrics["average_odds_difference"] = average_odds_difference(
            y_true=y_true, y_pred=y_pred, prot_attr=prot_attr, priv_group=priv_group,
            pos_label=pos_label, sample_weight=sample_weight
        )
        if len(set(y_pred)) > 1:
            group_metrics["conditional_demographic_disparity"] = conditional_demographic_disparity(
                y_true=y_true, y_pred=y_pred, prot_attr=prot_attr,
                pos_label=pos_label, sample_weight=sample_weight
            )
        else:
            group_metrics["conditional_demographic_disparity"] = None
        group_metrics["smoothed_edf"] = smoothed_edf(
            y_true=y_true, y_pred=y_pred, prot_attr=prot_attr,
            pos_label=pos_label, sample_weight=sample_weight
        )
        group_metrics["df_bias_amplification"] = df_bias_amplification(
            y_true=y_true, y_pred=y_pred, prot_attr=prot_attr,
            pos_label=pos_label, sample_weight=sample_weight
        )
        group_metrics["balanced_accuracy_score"] = balanced_accuracy_score(
            y_true=y_true, y_pred=y_pred, sample_weight=sample_weight
        )
    return group_metrics

print("✓ Fonction get_metrics définie.")

✓ Fonction get_metrics définie.


In [81]:
def compute_reweighing_weights(df_in, label_col, protected_col):
    """
    Calcule les poids de reweighing pour rendre le label indépendant de l'attribut sensible.

    Formule : W(S=s, Y=y) = P(Y=y) × P(S=s) / P(Y=y, S=s)
    """
    n = len(df_in)
    weights = pd.Series(1.0, index=df_in.index)

    for s_val in df_in[protected_col].unique():
        for y_val in df_in[label_col].unique():
            mask = (df_in[protected_col] == s_val) & (df_in[label_col] == y_val)
            n_sy = mask.sum()
            n_s = (df_in[protected_col] == s_val).sum()
            n_y = (df_in[label_col] == y_val).sum()

            if n_sy > 0:
                w = (n_y / n) * (n_s / n) / (n_sy / n)
                weights[mask] = w

    return weights



def apply_weights_to_csv(df_source, csv_out, weights, weights_col='WEIGHTS'):
    """Sauvegarde un CSV avec pondération sur le train uniquement et poids neutres sur valid/test."""
    df_tmp = df_source.copy()
    df_tmp[weights_col] = 1.0

    if len(weights) == len(df_tmp):
        df_tmp[weights_col] = weights.values
    else:
        train_mask = df_tmp['train_valid'] == 'train'
        if train_mask.sum() != len(weights):
            raise ValueError(
                f"Longueur des poids incompatible: {len(weights)} vs {train_mask.sum()} lignes train"
            )
        df_tmp.loc[train_mask, weights_col] = weights.values

    df_tmp.to_csv(csv_out, index=False)
    print(f"  CSV sauvegardé : {csv_out}")
    print(f"  Poids — min={df_tmp[weights_col].min():.4f}, max={df_tmp[weights_col].max():.4f}, mean={df_tmp[weights_col].mean():.4f}")
    return csv_out

print("✓ Fonctions compute_reweighing_weights et apply_weights_to_csv définies.")

✓ Fonctions compute_reweighing_weights et apply_weights_to_csv définies.


In [82]:
def prepare_predictions(preds_csv_path):
    """
    Charge le CSV de prédictions et prépare les colonnes binaires pour l'analyse.

    Returns:
        DataFrame enrichi (label_binary, preds_binary, Gender_Binary, Position_Binary, score_malade)
    """
    df_p = pd.read_csv(preds_csv_path)

    # Encodage binaire
    df_p['label_binary'] = (df_p['labels'] == 'malade').astype(int)
    df_p['preds_binary'] = (df_p['preds'] == 'malade').astype(int)
    df_p['Gender_Binary'] = (df_p['Patient Gender'] == 'M').astype(int)
    df_p['Position_Binary'] = (df_p['View Position'] == 'PA').astype(int)

    # Conversion logits → probabilités via softmax
    logit_cols = [c for c in df_p.columns if c.startswith('preds_logit')]
    if len(logit_cols) >= 2:
        logits = torch.tensor(df_p[logit_cols].values.astype(float))
        probs = torch.softmax(logits, dim=1).numpy()
        df_p['score_malade'] = probs[:, 0]
        df_p['score_sain'] = probs[:, 1]

    return df_p


def evaluate_model(df_p, split='valid', prot_attr='Gender_Binary', priv_group=1, label=''):
    """
    Évalue les métriques de fairness et performance sur un split donné.

    Returns:
        dict de métriques
    """
    df_eval = df_p[df_p['train_valid'] == split].copy()

    metrics = get_metrics(
        y_true=df_eval['label_binary'].values,
        y_pred=df_eval['preds_binary'].values,
        prot_attr=df_eval[prot_attr].values,
        priv_group=priv_group,
        pos_label=1,
    )
    metrics['method'] = label
    metrics['protected_attribute'] = prot_attr

    return metrics


def print_metrics(metrics, title=""):
    """Affiche proprement un dictionnaire de métriques."""
    if title:
        print(f"\n{'=' * 60}")
        print(title)
        print('=' * 60)
    for k, v in metrics.items():
        if isinstance(v, float):
            print(f"  {k:45s} : {v:+.4f}" if 'ratio' not in k else f"  {k:45s} : {v:.4f}")


def create_aif360_dataset(df_eval, label_col='label_binary', prot_attr='Gender_Binary', score_col=None):
    """Crée un BinaryLabelDataset AIF360 à partir d'un DataFrame."""
    df_aif = df_eval[[label_col, prot_attr]].copy().reset_index(drop=True)
    dataset = BinaryLabelDataset(
        df=df_aif,
        label_names=[label_col],
        protected_attribute_names=[prot_attr],
        favorable_label=1.0,
        unfavorable_label=0.0,
    )
    if score_col is not None and score_col in df_eval.columns:
        dataset.scores = df_eval[[score_col]].values.astype(float)
    return dataset


def fit_postprocessor_on_validation(df_val, method='ROC', prot_attr='Gender_Binary'):
    """Ajuste le post-processing sur la validation."""
    dataset_true = create_aif360_dataset(df_val, 'label_binary', prot_attr, score_col='score_malade')
    dataset_pred = dataset_true.copy(deepcopy=True)
    dataset_pred.labels = df_val['preds_binary'].values.reshape(-1, 1).astype(float)
    dataset_pred.scores = df_val['score_malade'].values.reshape(-1, 1).astype(float)

    privileged_groups = [{prot_attr: 1}]
    unprivileged_groups = [{prot_attr: 0}]

    if method == 'ROC':
        postproc = RejectOptionClassification(
            unprivileged_groups=unprivileged_groups,
            privileged_groups=privileged_groups,
            low_class_thresh=0.01,
            high_class_thresh=0.99,
            num_class_thresh=100,
            num_ROC_margin=50,
            metric_name="Statistical parity difference",
            metric_ub=0.05,
            metric_lb=-0.05,
        )
        postproc = postproc.fit(dataset_true, dataset_pred)
        print(f"  ROC — Seuil optimal : {postproc.classification_threshold:.4f}")
        print(f"  ROC — Marge optimale : {postproc.ROC_margin:.4f}")
    elif method == 'CalibratedEqOdds':
        postproc = CalibratedEqOddsPostprocessing(
            privileged_groups=privileged_groups,
            unprivileged_groups=unprivileged_groups,
            cost_constraint='fnr',
            seed=42,
        )
        postproc = postproc.fit(dataset_true, dataset_pred)
    else:
        raise ValueError(f"Méthode de post-processing inconnue: {method}")

    return postproc


def apply_postprocessor_to_split(df_eval, postproc, prot_attr='Gender_Binary'):
    """Applique un post-processing déjà ajusté sur un split donné."""
    dataset_true = create_aif360_dataset(df_eval, 'label_binary', prot_attr, score_col='score_malade')
    dataset_pred = dataset_true.copy(deepcopy=True)
    dataset_pred.labels = df_eval['preds_binary'].values.reshape(-1, 1).astype(float)
    dataset_pred.scores = df_eval['score_malade'].values.reshape(-1, 1).astype(float)
    dataset_corrected = postproc.predict(dataset_pred)
    return dataset_corrected.labels[:, 0]


def evaluate_post_processing(df_val, df_eval, method='ROC', prot_attr='Gender_Binary', label=''):
    """Ajuste sur validation puis évalue sur le split d'évaluation fourni."""
    postproc = fit_postprocessor_on_validation(df_val, method=method, prot_attr=prot_attr)
    corrected_preds = apply_postprocessor_to_split(df_eval, postproc, prot_attr=prot_attr)

    metrics = get_metrics(
        y_true=df_eval['label_binary'].values,
        y_pred=corrected_preds,
        prot_attr=df_eval[prot_attr].values,
        priv_group=1,
        pos_label=1,
    )
    metrics['method'] = label
    metrics['protected_attribute'] = prot_attr

    return metrics, postproc

print("✓ Fonctions prepare_predictions, evaluate_model, print_metrics, create_aif360_dataset et post-processing définies.")

✓ Fonctions prepare_predictions, evaluate_model, print_metrics, create_aif360_dataset et post-processing définies.


### 1.4 Synthèse exploratoire

Au terme de cette première partie, on peut retenir quatre points simples :
- le dataset est exploitable après nettoyage de l'âge aberrant et la suppression de la colonne vide ;
- la distribution globale de la cible est assez équilibrée, sans déséquilibre massif ;
- la position de vue est la variable la plus suspecte sur le plan des biais techniques ;
- le genre présente un écart plus faible, mais il doit quand même être contrôlé par les métriques de fairness.

Cette lecture exploratoire donne une base claire pour la suite : on ne corrige pas tout, on cible d'abord les variables qui ont un vrai potentiel de biais structurel. C'est ce qui justifie ensuite l'usage de méthodes de mitigation comme le reweighing et les post-processings dans la partie II.

## II/- Analyse de l'impact de la pondération sur les performances du modèle

Cette partie évalue l'effet des différentes stratégies de pondération sur le compromis entre performance et équité.

Plus précisément, nous cherchons à répondre à trois questions :
- la pondération améliore-t-elle la fairness sans trop dégrader la performance ?
- quelles méthodes corrigent le mieux les biais observés en exploration ?
- quel compromis retenir entre Balanced Accuracy et métriques d'équité ?

Nous commençons par une baseline sans pondération, puis nous testons plusieurs variantes de Reweighing avant de comparer ces résultats aux approches de post-processing.

### 2.1 Baseline de référence (sans pondération)

In [83]:
# recharge le module d'entrainement
import importlib
import train_classifieur as tc

importlib.reload(tc)

# Rebind explicite des objets utilises dans le notebook
ChestXRayClassifier = tc.ChestXRayClassifier
pred_classifier = tc.pred_classifier
preds_todf = tc.preds_todf
train_classifier = tc.train_classifier
transforms_valid = tc.transforms_valid

#### Entraînement du modèle

In [84]:
# === Entraînement du modèle Baseline (WEIGHTS = 1 pour tout le monde) ===

dt = datetime.now()
logdir_baseline = f"{EXPE_DIR}/baseline_{dt.strftime('%Y_%m_%d_%H_%M_%S')}"

ckpt_path_baseline, ckpt_score_baseline = train_classifier(
    logdir=logdir_baseline,
    datadir=DATA_DIR,
    csv=CSV_PATH,
    weights_col="WEIGHTS",
    max_epochs=MAX_EPOCHS,
)

print(f"\nModèle sauvegardé : {ckpt_path_baseline}")
print(f"Meilleur score (val_loss) : {ckpt_score_baseline:.4f}")

c:\Users\remim\Documents\Projet Fairness\Fairness-in-AI\Data\metadata.csv C:\Users\remim\AppData\Local\Temp\fairness_runs/baseline_2026_04_03_06_07_23/csv_in_WEIGHTS.csv
num_workers set to : 0


GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


num_workers set to : 0
batch_size set to : 16
Start training


┏━━━┳━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name  ┃ Type                  ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model │ ResNet                │ 11.2 M │ train │     0 │
│ 1 │ cm    │ BinaryConfusionMatrix │      0 │ train │     0 │
└───┴───────┴───────────────────────┴────────┴───────┴───────┘

Trainable params: 11.2 M                                                                                           
Non-trainable params: 0                                                                                            
Total params: 11.2 M                                                                                               
Total estimated model params size (MB): 44                                                                         
Modules in train mode: 69                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

`Trainer.fit` stopped: `max_epochs=1` reached.


End of training 290.2188560962677

Modèle sauvegardé : C:\Users\remim\AppData\Local\Temp\fairness_runs\baseline_2026_04_03_06_07_23\best-val-loss.ckpt
Meilleur score (val_loss) : 0.6313


#### Génération des prédictions

In [85]:
# === Predictions du modele Baseline sur validation ===
BASELINE_VAL_PREDS = f"{logdir_baseline}/preds_baseline_valid.csv"

df_baseline = predict_partition(CSV_PATH, DATA_DIR, 'valid', ckpt_path_baseline, preds_col='preds')
df_baseline_val = df_baseline.copy()

df_baseline.to_csv(BASELINE_VAL_PREDS, index=False)
print(f"\nPredictions validation sauvegardees : {BASELINE_VAL_PREDS}")

num_workers set to : 0


100%|██████████| 78/78 [00:32<00:00,  2.42it/s]


Predictions validation sauvegardees : C:\Users\remim\AppData\Local\Temp\fairness_runs/baseline_2026_04_03_06_07_23/preds_baseline_valid.csv


#### Evaluation du modèle

In [86]:
# === Évaluation du modèle Baseline ===
# Balanced Accuracy globale sur la validation
ba_baseline = balanced_accuracy_score(df_baseline['label_binary'], df_baseline['preds_binary'])
ba_baseline_val = balanced_accuracy_score(df_baseline_val['label_binary'], df_baseline_val['preds_binary'])
print(f"Balanced Accuracy (validation) : {ba_baseline_val:.4f}")
print(f"Balanced Accuracy (valid)       : {ba_baseline:.4f}")

# Métriques par Genre
metrics_baseline_gender = evaluate_model(df_baseline, split='valid', prot_attr='Gender_Binary', label='Baseline')
print_metrics(metrics_baseline_gender, "Baseline — Fairness par Genre (valid)")

# Métriques par Position
metrics_baseline_pos = evaluate_model(df_baseline, split='valid', prot_attr='Position_Binary', label='Baseline (pos)')
print_metrics(metrics_baseline_pos, "Baseline — Fairness par Position (valid)")

Balanced Accuracy (validation) : 0.6722
Balanced Accuracy (valid)       : 0.6722

Baseline — Fairness par Genre (valid)
  base_rate_truth                               : +0.4326
  statistical_parity_difference                 : +0.0399
  disparate_impact_ratio                        : 1.1147
  base_rate_preds                               : +0.3668
  equal_opportunity_difference                  : -0.0346
  average_odds_difference                       : +0.0135
  conditional_demographic_disparity             : +0.0032
  smoothed_edf                                  : +0.1084
  df_bias_amplification                         : -0.0282
  balanced_accuracy_score                       : +0.6722

Baseline — Fairness par Position (valid)
  base_rate_truth                               : +0.4326
  statistical_parity_difference                 : +0.1333
  disparate_impact_ratio                        : 1.4218
  base_rate_preds                               : +0.3668
  equal_opportunity_differen

##### Analyse de la Baseline
1. **Performance globale**
- **Balanced Accuracy = 0.5809**: 0.6174 : Performance modérée. C'est notre score de référence, et il montre que le modèle capture environ 62 % du signal clinique équitablement entre les deux classes.

2. **Fairness par genre** (Hommes/Femmes)
-SPD = +0.0378 : Très léger écart de parité statistique (< 4 %)
-DIR = 1.1525 : Dans la zone acceptable [0.8 ; 1.25]. Le modèle favorise très légèrement les hommes, mais l'écart reste mineur.
- EOD = +0.0007 : Quasi inexistant, les taux de détection vrais positifs sont équilibrés entre genres.
- AOD = +0.0214 : Très faible moyenne des erreurs de type I et II.

==> Conclusion : Le modèle est essentiellement équitable par rapport au genre.

3. **Fairness par position** (PA/AP)
- SPD = +0.0582 : Ecart modéré de parité statistique (≈ 6 %)
- DIR = 1.2388 : À la limite haute de la zone acceptable [0.8 ; 1.25], proche d'en sortir. Le modèle favorise légèrement une position AP.
- EOD = +0.0345 : Le taux de détection des vrais positifs diffère selon la position. L'écart étant positif, le modèle identifie les patients réellement malades avec un taux de succès supérieur d'environ 3,45% sur les vues AP par rapport aux vues PA. Cependant, cet écart relativement faible, mis en perspective avec notre DIR (1.23), confirme surtout que le modèle a tendance à prédire la classe positive beaucoup plus facilement pour la vue AP, augmentant mécaniquement son taux de Vrais Positifs mais aussi son taux de Faux Positifs
- Bias Amplification = +0.1598 : Amplification notable du biais au travers du modèle.

==> Conclusion : Biais modéré sur la position, le modèle utilise potentiellement la position comme raccourci prédictif.

4. **Lecture d’ensemble**
- Le baseline montre un compromis intéressant mais imparfait :
    - Genre bien contrôlé (DIR = 1.15, acceptable)
    - Position proche du seuil critique (DIR = 1.24, doit être réduit)
    - Performance correcte (balanced_accuracy = 0.62) mais améliorable
    
5. Implication pour la Reweighing
Priorité : Tester RW Position et RW Genre×Position pour ramener DIR Position vers 1.0
Objectif : Réduire le DIR position sans dégrader la BA et sans sacrifier l'équité genre
Attendre : Les résultats des variantes pour voir quel est le meilleur compromis performance/équité


### 2.2 Reweighing par genre

In [87]:
# === Calcul des poids de Reweighing (Genre) ===
print("Poids de Reweighing par Genre :")
df_train = df_split[df_split['train_valid'] == 'train'].copy()
weights_gender = compute_reweighing_weights(df_train, 'label_binary', 'Gender_Binary')

for g in [0, 1]:
    for l in [0, 1]:
        mask = (df_train['Gender_Binary'] == g) & (df_train['label_binary'] == l)
        g_name = 'M' if g == 1 else 'F'
        l_name = 'malade' if l == 1 else 'sain'
        if mask.any():
            print(f"  Genre={g_name}, Label={l_name} : poids={weights_gender[mask].iloc[0]:.4f} (n={mask.sum()})")
        else:
            print(f"  Genre={g_name}, Label={l_name} : combinaison absente du train")

CSV_GENDER = f"{WORK_DIR}/metadata_rw_gender.csv"
apply_weights_to_csv(df_split, CSV_GENDER, weights_gender)

Poids de Reweighing par Genre :
  Genre=F, Label=sain : poids=0.9310 (n=892)
  Genre=F, Label=malade : poids=1.0958 (n=643)
  Genre=M, Label=sain : poids=1.0489 (n=1259)
  Genre=M, Label=malade : poids=0.9479 (n=1182)
  CSV sauvegardé : C:\Users\remim\AppData\Local\Temp\fairness_runs/metadata_rw_gender.csv
  Poids — min=0.9310, max=1.0958, mean=1.0000


'C:\\Users\\remim\\AppData\\Local\\Temp\\fairness_runs/metadata_rw_gender.csv'

#### Entrainement du modèle

In [88]:
# === Entraînement avec poids Reweighing Genre ===
dt = datetime.now()
logdir_gender = f"{EXPE_DIR}/rw_gender_{dt.strftime('%Y_%m_%d_%H_%M_%S')}"

ckpt_path_gender, ckpt_score_gender = train_classifier(
    logdir=logdir_gender,
    datadir=DATA_DIR,
    csv=CSV_GENDER,
    weights_col="WEIGHTS",
    max_epochs=MAX_EPOCHS,
)
print(f"\nModèle RW Genre sauvegardé : {ckpt_path_gender}")
print(f"Meilleur score (val_loss) : {ckpt_score_gender:.4f}")

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


C:\Users\remim\AppData\Local\Temp\fairness_runs/metadata_rw_gender.csv C:\Users\remim\AppData\Local\Temp\fairness_runs/rw_gender_2026_04_03_06_12_46/csv_in_WEIGHTS.csv
num_workers set to : 0
num_workers set to : 0
batch_size set to : 16
Start training


┏━━━┳━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name  ┃ Type                  ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model │ ResNet                │ 11.2 M │ train │     0 │
│ 1 │ cm    │ BinaryConfusionMatrix │      0 │ train │     0 │
└───┴───────┴───────────────────────┴────────┴───────┴───────┘

Trainable params: 11.2 M                                                                                           
Non-trainable params: 0                                                                                            
Total params: 11.2 M                                                                                               
Total estimated model params size (MB): 44                                                                         
Modules in train mode: 69                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

`Trainer.fit` stopped: `max_epochs=1` reached.


End of training 226.1787073612213

Modèle RW Genre sauvegardé : C:\Users\remim\AppData\Local\Temp\fairness_runs\rw_gender_2026_04_03_06_12_46\best-val-loss.ckpt
Meilleur score (val_loss) : 0.6348


#### Génération des prédictions

In [89]:
# === Predictions du modele Reweighing Genre sur validation ===
GENDER_VAL_PREDS = f"{logdir_gender}/preds_rw_gender_valid.csv"

df_gender = predict_partition(CSV_GENDER, DATA_DIR, 'valid', ckpt_path_gender, preds_col='preds')
df_gender_val = df_gender.copy()

df_gender.to_csv(GENDER_VAL_PREDS, index=False)
print(f"\nPredictions validation sauvegardees : {GENDER_VAL_PREDS}")

num_workers set to : 0


100%|██████████| 78/78 [00:24<00:00,  3.21it/s]


Predictions validation sauvegardees : C:\Users\remim\AppData\Local\Temp\fairness_runs/rw_gender_2026_04_03_06_12_46/preds_rw_gender_valid.csv


#### Evaluation du modèle

In [90]:
# === Évaluation du modèle Reweighing Genre ===
metrics_rw_gender = evaluate_model(df_gender, split='valid', prot_attr='Gender_Binary', label='RW Genre')
print_metrics(metrics_rw_gender, "Reweighing Genre — Fairness par Genre (valid)")

# Aussi évaluer par position (effet croisé)
metrics_rw_gender_pos = evaluate_model(df_gender, split='valid', prot_attr='Position_Binary', label='RW Genre (pos)')
print_metrics(metrics_rw_gender_pos, "Reweighing Genre — Fairness par Position (valid)")


Reweighing Genre — Fairness par Genre (valid)
  base_rate_truth                               : +0.4326
  statistical_parity_difference                 : +0.0632
  disparate_impact_ratio                        : 1.2709
  base_rate_preds                               : +0.2624
  equal_opportunity_difference                  : +0.0047
  average_odds_difference                       : +0.0422
  conditional_demographic_disparity             : +0.0060
  smoothed_edf                                  : +0.2392
  df_bias_amplification                         : +0.1026
  balanced_accuracy_score                       : +0.6382

Reweighing Genre — Fairness par Position (valid)
  base_rate_truth                               : +0.4326
  statistical_parity_difference                 : +0.0599
  disparate_impact_ratio                        : 1.2501
  base_rate_preds                               : +0.2624
  equal_opportunity_difference                  : +0.0397
  average_odds_difference          

##### Analyse du Reweighing (Genre)
1. *Impact sur la performance* : la Balanced Accuracy passe de 0.6174 à 0.6293. C’est une petite amélioration, la pondération par genre rend le modèle légèrement meilleur sur la validation.

2. *Impact sur l'équité par genre:* le modèle était déjà assez proche de la parité, donc le gain reste modeste mais cohérent. Le DIR genre reste quasiment inchangé, de 1.1525 à 1.1515, tandis que le SPD baisse un peu de +0.0378 à +0.0331. Les métriques d’erreur vont aussi dans le bon sens avec un EOD qui devient légèrement négatif (-0.0069) et un AOD qui baisse à +0.0150.

3. *Lecture par position :* la pondération faite sur le genre a aussi un effet sur la position. Le DIR position descend de 1.2388 à 1.1429, donc on se rapproche clairement de la zone de parité. Le SPD position diminue aussi de +0.0582 à +0.0317. En revanche, l’EOD position monte à +0.0495, ce qui montre que tous les indicateurs ne s’améliorent pas dans le même sens.

4. *Bias amplification :* elle baisse nettement sur la position, de +0.1598 à +0.0796, ce qui indique que le modèle amplifie moins le biais qu’avant, même si l’effet n’a pas disparu.

==> Conclusion : Puisque le modèle de base était déjà proche de la parité sur le genre, cette pondération agit moie comme une régularisation du signal parasite. Elle parvient à :

- Améliorer légèrement la performance globale (Balanced Accuracy).

- Maintenir l'équité existante sur le genre.

- Réduire indirectement le biais technique lié à la position de vue (baisse du DIR et de l'amplification des biais).

Cependant, cette mitigation reste partielle. Pour corriger efficacement le biais d'acquisition matériel (PA vs AP), cibler le genre n'est pas la stratégie optimale. Ce constat justifie logiquement la suite de notre expérimentation : tester le RW Position puis le RW Croisé (Genre x Position) pour attaquer le déséquilibre à la racine.


### 2.3 Reweighing par position (PA/AP)

In [91]:
# === Calcul des poids de Reweighing (Position) ===
print("Poids de Reweighing par Position :")
df_train = df_split[df_split['train_valid'] == 'train'].copy()
weights_position = compute_reweighing_weights(df_train, 'label_binary', 'Position_Binary')

for p in [0, 1]:
    for l in [0, 1]:
        mask = (df_train['Position_Binary'] == p) & (df_train['label_binary'] == l)
        p_name = 'PA' if p == 1 else 'AP'
        l_name = 'malade' if l == 1 else 'sain'
        if mask.any():
            print(f"  Position={p_name}, Label={l_name} : poids={weights_position[mask].iloc[0]:.4f} (n={mask.sum()})")
        else:
            print(f"  Position={p_name}, Label={l_name} : combinaison absente du train")

CSV_POSITION = f"{WORK_DIR}/metadata_rw_position.csv"
apply_weights_to_csv(df_split, CSV_POSITION, weights_position)

Poids de Reweighing par Position :
  Position=AP, Label=sain : poids=1.1441 (n=732)
  Position=AP, Label=malade : poids=0.8708 (n=816)
  Position=PA, Label=sain : poids=0.9257 (n=1419)
  Position=PA, Label=malade : poids=1.1045 (n=1009)
  CSV sauvegardé : C:\Users\remim\AppData\Local\Temp\fairness_runs/metadata_rw_position.csv
  Poids — min=0.8708, max=1.1441, mean=1.0000


'C:\\Users\\remim\\AppData\\Local\\Temp\\fairness_runs/metadata_rw_position.csv'

#### Entrainement du modèle

In [92]:
# === Entraînement avec poids Reweighing Position ===
dt = datetime.now()
logdir_pos = f"{EXPE_DIR}/rw_position_{dt.strftime('%Y_%m_%d_%H_%M_%S')}"

ckpt_path_pos, ckpt_score_pos = train_classifier(
    logdir=logdir_pos,
    datadir=DATA_DIR,
    csv=CSV_POSITION,
    weights_col="WEIGHTS",
    max_epochs=MAX_EPOCHS,
)
print(f"\nModèle RW Position sauvegardé : {ckpt_path_pos}")
print(f"Meilleur score (val_loss) : {ckpt_score_pos:.4f}")

C:\Users\remim\AppData\Local\Temp\fairness_runs/metadata_rw_position.csv C:\Users\remim\AppData\Local\Temp\fairness_runs/rw_position_2026_04_03_06_16_57/csv_in_WEIGHTS.csv
num_workers set to : 0


GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


num_workers set to : 0
batch_size set to : 16
Start training


┏━━━┳━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name  ┃ Type                  ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model │ ResNet                │ 11.2 M │ train │     0 │
│ 1 │ cm    │ BinaryConfusionMatrix │      0 │ train │     0 │
└───┴───────┴───────────────────────┴────────┴───────┴───────┘

Trainable params: 11.2 M                                                                                           
Non-trainable params: 0                                                                                            
Total params: 11.2 M                                                                                               
Total estimated model params size (MB): 44                                                                         
Modules in train mode: 69                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

`Trainer.fit` stopped: `max_epochs=1` reached.


End of training 223.04234743118286

Modèle RW Position sauvegardé : C:\Users\remim\AppData\Local\Temp\fairness_runs\rw_position_2026_04_03_06_16_57\best-val-loss.ckpt
Meilleur score (val_loss) : 0.7185


#### Génération des prédictions

In [93]:
# === Prédictions du modèle Reweighing Position sur validation ===
POS_VAL_PREDS = f"{logdir_pos}/preds_rw_position_valid.csv"

df_pos = predict_partition(CSV_POSITION, DATA_DIR, 'valid', ckpt_path_pos, preds_col='preds')
df_pos_val = df_pos.copy()

df_pos.to_csv(POS_VAL_PREDS, index=False)
print(f"\nPredictions validation sauvegardees : {POS_VAL_PREDS}")

num_workers set to : 0


100%|██████████| 78/78 [00:25<00:00,  3.09it/s]


Predictions validation sauvegardees : C:\Users\remim\AppData\Local\Temp\fairness_runs/rw_position_2026_04_03_06_16_57/preds_rw_position_valid.csv


#### Evaluation du modèle

In [94]:
# === Évaluation du modèle Reweighing Position ===
metrics_rw_pos = evaluate_model(df_pos, split='valid', prot_attr='Position_Binary', label='RW Position')
print_metrics(metrics_rw_pos, "Reweighing Position — Fairness par Position (valid)")

# Aussi évaluer par genre (effet croisé)
metrics_rw_pos_gender = evaluate_model(df_pos, split='valid', prot_attr='Gender_Binary', label='RW Position (genre)')
print_metrics(metrics_rw_pos_gender, "Reweighing Position — Fairness par Genre (valid)")


Reweighing Position — Fairness par Position (valid)
  base_rate_truth                               : +0.4326
  statistical_parity_difference                 : -0.1182
  disparate_impact_ratio                        : 0.8020
  base_rate_preds                               : +0.5522
  equal_opportunity_difference                  : -0.1346
  average_odds_difference                       : -0.1252
  conditional_demographic_disparity             : -0.0270
  smoothed_edf                                  : +0.2570
  df_bias_amplification                         : +0.2026
  balanced_accuracy_score                       : +0.6216

Reweighing Position — Fairness par Genre (valid)
  base_rate_truth                               : +0.4326
  statistical_parity_difference                 : -0.0213
  disparate_impact_ratio                        : 0.9621
  base_rate_preds                               : +0.5522
  equal_opportunity_difference                  : -0.0740
  average_odds_difference    

##### Analyse du Reweighing (Position)

1. **Impact sur la performance**
- La **Balanced Accuracy** passe de **0.6174** à **0.5587**.
- C’est une baisse nette, donc cette pondération corrige le biais de position au prix d’une dégradation importante de la capacité prédictive.
- Autrement dit, cette variante est plus agressive que RW Genre et pénalise davantage l’utilité clinique.

2. **Impact sur l’équité par position**
- Le **DIR position** passe de **1.2388** à **0.7907**.
- On sort de la zone haute observée sur la baseline, mais on tombe maintenant légèrement sous le seuil classique de tolérance autour de 0.8.
- Le **SPD** devient **-0.1866**, ce qui montre une inversion marquée du déséquilibre initial.
- Les métriques d’erreur suivent la même logique:
  - **EOD = -0.1089**
  - **AOD = -0.1806**
- Cela suggère que la correction ne se contente pas de réduire le biais : elle le **sur-corrige** en partie.

3. **Lecture sur le genre**
- Sur le genre, le modèle reste plutôt raisonnable:
  - **DIR genre = 1.0805**
  - **SPD genre = +0.0637**
- C’est mieux centré que la baseline sur le ratio, mais pas spectaculaire.
- En revanche, les métriques d’erreur restent moins propres qu’avec RW Genre:
  - **EOD = +0.0099**
  - **AOD = +0.0516**
- Donc RW Position ne dégrade pas complètement l’équité genre, mais ne l’améliore pas franchement non plus.

4. **Bias amplification**
- La **bias amplification** sur la position devient très élevée: **+0.9410**.
- C’est un signal important: le modèle ne se contente pas d’être imparfait, il produit une distribution de prédictions fortement déséquilibrée.
- Cela confirme que la pondération par position est trop forte dans cette configuration.

5. Interprétation

Ce résultat montre que **RW Position corrige le biais position de manière trop radicale**.  
On obtient bien un déplacement du DIR vers la zone de parité, mais au prix d’une chute de performance et d’un déséquilibre inversé sur certaines métriques.

En pratique, cela veut dire que:
- la méthode est utile pour tester la sensibilité du modèle à la position,
- mais elle n’offre pas ici le meilleur compromis performance/équité,
- et elle semble moins stable que RW Genre.

==> Conclusion

**RW Position améliore la fairness sur la position, mais dégrade trop la Balanced Accuracy et semble sur-corriger le biais.**  
C’est donc une mitigation informative, mais pas forcément la meilleure candidate finale. Elle justifie en revanche de tester ensuite **RW Genre × Position**, qui peut répartir la correction de façon plus fine.

### 2.4 Reweighing croise genre x position

#### Entraînement + Prédictions + Evaluation

In [95]:
# === Reweighing croisé: poids + entraînement + prédiction + évaluation ===
df_train = df_split[df_split['train_valid'] == 'train'].copy()

# 0=F_AP, 1=F_PA, 2=M_AP, 3=M_PA
df_train['GenderPosition_Binary'] = 2 * df_train['Gender_Binary'] + df_train['Position_Binary']
weights_cross = compute_reweighing_weights(df_train, 'label_binary', 'GenderPosition_Binary')
CSV_CROSS = f"{WORK_DIR}/metadata_rw_gender_position.csv"
apply_weights_to_csv(df_split, CSV_CROSS, weights_cross)

dt = datetime.now()
logdir_cross = f"{EXPE_DIR}/rw_gender_position_{dt.strftime('%Y_%m_%d_%H_%M_%S')}"
ckpt_path_cross, ckpt_score_cross = train_classifier(
    logdir=logdir_cross,
    datadir=DATA_DIR,
    csv=CSV_CROSS,
    weights_col="WEIGHTS",
    max_epochs=MAX_EPOCHS,
)
print(f"\nModèle RW croisé sauvegardé : {ckpt_path_cross}")
print(f"Meilleur score (val_loss) : {ckpt_score_cross:.4f}")

CROSS_VAL_PREDS = f"{logdir_cross}/preds_rw_gender_position_valid.csv"

df_cross = predict_partition(CSV_CROSS, DATA_DIR, 'valid', ckpt_path_cross, preds_col='preds')
df_cross_val = df_cross.copy()

df_cross.to_csv(CROSS_VAL_PREDS, index=False)
print(f"\nPredictions validation sauvegardees : {CROSS_VAL_PREDS}")

metrics_rw_cross_gender = evaluate_model(df_cross, split='valid', prot_attr='Gender_Binary', label='RW Genre×Position')
metrics_rw_cross_pos = evaluate_model(df_cross, split='valid', prot_attr='Position_Binary', label='RW Genre×Position')
print_metrics(metrics_rw_cross_gender, "RW Genre×Position — Fairness par Genre (valid)")
print_metrics(metrics_rw_cross_pos, "RW Genre×Position — Fairness par Position (valid)")

  CSV sauvegardé : C:\Users\remim\AppData\Local\Temp\fairness_runs/metadata_rw_gender_position.csv
  Poids — min=0.8295, max=1.2112, mean=1.0000
C:\Users\remim\AppData\Local\Temp\fairness_runs/metadata_rw_gender_position.csv C:\Users\remim\AppData\Local\Temp\fairness_runs/rw_gender_position_2026_04_03_06_21_06/csv_in_WEIGHTS.csv
num_workers set to : 0


GPU available: False, used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


num_workers set to : 0
batch_size set to : 16
Start training


┏━━━┳━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name  ┃ Type                  ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model │ ResNet                │ 11.2 M │ train │     0 │
│ 1 │ cm    │ BinaryConfusionMatrix │      0 │ train │     0 │
└───┴───────┴───────────────────────┴────────┴───────┴───────┘

Trainable params: 11.2 M                                                                                           
Non-trainable params: 0                                                                                            
Total params: 11.2 M                                                                                               
Total estimated model params size (MB): 44                                                                         
Modules in train mode: 69                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

`Trainer.fit` stopped: `max_epochs=1` reached.


End of training 223.7788965702057

Modèle RW croisé sauvegardé : C:\Users\remim\AppData\Local\Temp\fairness_runs\rw_gender_position_2026_04_03_06_21_06\best-val-loss.ckpt
Meilleur score (val_loss) : 0.6553
num_workers set to : 0


100%|██████████| 78/78 [00:24<00:00,  3.23it/s]


Predictions validation sauvegardees : C:\Users\remim\AppData\Local\Temp\fairness_runs/rw_gender_position_2026_04_03_06_21_06/preds_rw_gender_position_valid.csv

RW Genre×Position — Fairness par Genre (valid)
  base_rate_truth                               : +0.4326
  statistical_parity_difference                 : +0.1170
  disparate_impact_ratio                        : 1.2315
  base_rate_preds                               : +0.5594
  equal_opportunity_difference                  : +0.0303
  average_odds_difference                       : +0.0926
  conditional_demographic_disparity             : +0.0087
  smoothed_edf                                  : +0.2691
  df_bias_amplification                         : +0.1325
  balanced_accuracy_score                       : +0.6414

RW Genre×Position — Fairness par Position (valid)
  base_rate_truth                               : +0.4326
  statistical_parity_difference                 : -0.0720
  disparate_impact_ratio                     

##### Analyse du Reweighing (Genre × Position)

1. **Impact sur la performance**
- La **Balanced Accuracy** passe de **0.6174** (baseline) à **0.5710**.
- C’est une baisse notable de performance.
- Le modèle croisé fait un peu mieux que **RW Position** (0.5587), mais reste clairement en dessous de la baseline et de **RW Genre** (0.6293).

2. **Impact sur l’équité par genre**
- **SPD genre = +0.0365**: écart faible, proche de la baseline.
- **DIR genre = 1.0525**: très proche de 1, donc bon niveau de parité.
- **EOD = +0.0237** et **AOD = +0.0277**: écarts présents mais modérés.
- **df_bias_amplification = -0.0088**: point positif, le modèle n’amplifie pas le biais genre (légère réduction).

3. **Impact sur l’équité par position**
- **SPD position = -0.2799**: écart très important, avec inversion forte du biais.
- **DIR position = 0.6587**: nettement hors zone usuelle de parité.
- **EOD position = -0.2670**: différence marquée des taux de vrais positifs entre groupes de position.
- **smoothed_edf = +0.9357** et **df_bias_amplification = +0.8813**: signaux d’iniquité très élevés.
- En plus, **base_rate_preds = 0.7135** suggère une tendance forte à prédire la classe positive, ce qui peut dégrader le compromis global.

4. **Lecture d’ensemble**
- Sur le **genre**, la méthode est plutôt correcte.
- Sur la **position**, elle est clairement trop agressive et produit une sur-correction plus marquée encore que RW Position.
- Le compromis final est donc défavorable: perte de performance + iniquité position importante.

==> Conclusion

Finalement, RW Genre×Position améliore surtout la parité sur le genre, mais dégrade fortement l’équité sur la position et réduit la performance globale.  



### 2.5 Comparaison des méthodes de pondération (balanced accuracy + fairness)

In [96]:
# === Tableau comparatif Pre-processing ===
required = ['df_baseline', 'df_gender', 'df_pos', 'df_cross']
missing = [v for v in required if v not in globals()]
if missing:
    raise RuntimeError(f"Variables manquantes pour la comparaison pre-processing: {missing}")

all_pre_metrics = []
for df_pred, name, prot in [
    (df_baseline, 'Baseline', 'Gender_Binary'),
    (df_baseline, 'Baseline', 'Position_Binary'),
    (df_gender, 'RW Genre', 'Gender_Binary'),
    (df_gender, 'RW Genre', 'Position_Binary'),
    (df_pos, 'RW Position', 'Gender_Binary'),
    (df_pos, 'RW Position', 'Position_Binary'),
    (df_cross, 'RW Genre×Position', 'Gender_Binary'),
    (df_cross, 'RW Genre×Position', 'Position_Binary'),
]:
    m = evaluate_model(df_pred, split='valid', prot_attr=prot, label=name)
    m['attr'] = 'Genre' if 'Gender' in prot else 'Position'
    all_pre_metrics.append(m)

df_pre = pd.DataFrame(all_pre_metrics)
cols = [
    'method', 'attr', 'balanced_accuracy_score',
    'statistical_parity_difference', 'disparate_impact_ratio',
    'equal_opportunity_difference', 'average_odds_difference'
]

print("=== Tableau comparatif — Pre-processing ===")
display(df_pre[cols].sort_values(['attr', 'method']).round(4))

=== Tableau comparatif — Pre-processing ===


,method,attr,balanced_accuracy_score,statistical_parity_difference,disparate_impact_ratio,equal_opportunity_difference,average_odds_difference
0,Baseline,Genre,0.6722,0.0399,1.1147,-0.0346,0.0135
2,RW Genre,Genre,0.6382,0.0632,1.2709,0.0047,0.0422
6,RW Genre×Position,Genre,0.6414,0.1170,1.2315,0.0303,0.0926
4,RW Position,Genre,0.6216,-0.0213,0.9621,-0.0740,-0.0402
1,Baseline,Position,0.6722,0.1333,1.4218,0.0919,0.1216
3,RW Genre,Position,0.6382,0.0599,1.2501,0.0397,0.0519
7,RW Genre×Position,Position,0.6414,-0.0720,0.8773,-0.0825,-0.0792
5,RW Position,Position,0.6216,-0.1182,0.8020,-0.1346,-0.1252


4 constats ressortent :

1. **RW Genre est la meilleure option globale en pre-processing.**  

meilleure performance (**Balanced Accuracy (BA) = 0.6293**) tout en améliorant la fairness sur les deux attributs par rapport à la baseline:
- Genre: SPD passe de 0.0378 à 0.0331, AOD de 0.0214 à 0.0150.
- Position: SPD passe de 0.0582 à 0.0317, DIR de 1.2388 à 1.1429, AOD de 0.0506 à 0.0282.

2. **Sur le genre, la baseline était déjà correcte, donc les gains restent modestes.**  

Les écarts de fairness genre sont déjà faibles en baseline. RW Genre apporte une amélioration légère mais cohérente sans coût de performance, alors que RW Position et RW Genre×Position dégradent davantage la BA pour un bénéfice limité sur cet attribut.

3. **Sur la position, RW Position et RW Genre×Position sur-corrigent.**  

Les deux méthodes inversent fortement le biais:
- RW Position: SPD = -0.1866, DIR = 0.7907.
- RW Genre×Position: SPD = -0.2799, DIR = 0.6587.
Cette sur-correction s’accompagne d’une baisse nette de performance (BA = 0.5587 et 0.5710), ce qui rend le compromis défavorable.

4. **Lecture finale du compromis performance/fairness (pre-processing uniquement).**  

- **RW Genre**: meilleur compromis global.
- **Baseline**: point de référence acceptable mais moins équilibré sur la position.
- **RW Position** et **RW Genre×Position**: méthodes informatives pour l’analyse, mais trop agressives dans cette exécution.

##### Conclusion 

Dans la suite du notebook, la stratégie la plus pertinente est de garder **RW Genre** comme base de pre-processing, puis de tester le post-processing pour cibler plus finement les écarts restants, en particulier sur la position de vue.

## III/- Analyse de l'impact du post-processing

Selon nos résultats de la partie II, il s'est avéré que **RW Genre** était le meilleur compromis performance/fairness en pre-processing. Nous testons à présent des stratégies de post-processing pour affiner la correction des biais résiduels, en particulier sur l'attribut **Position de vue**. Cette section évalue deux méthodes de calibration, **Reject Option Classification (ROC)** et **Calibrated Equalized Odds**, appliquées au modèle de base et au modèle amélioré par RW Genre, sur les deux attributs sensibles. L'objectif est d'identifier s'il existe une combinaison pré/post-processing supérieure à RW Genre seul, tout en contrôlant la dégradation de performance.

In [97]:
def apply_post_processing(df_val, df_eval, method='ROC', prot_attr='Gender_Binary', label=''):
    """
    Ajuste une méthode de post-processing sur la validation puis l'applique sur validation.

    Args:
        df_val: DataFrame de validation avec scores et prédictions
        df_eval: DataFrame de test avec scores et prédictions
        method: 'ROC' ou 'CalibratedEqOdds'
        prot_attr: attribut sensible binaire
        label: nom de la configuration

    Returns:
        dict de metriques sur le split d'evaluation
    """
    postproc = fit_postprocessor_on_validation(df_val, method=method, prot_attr=prot_attr)
    corrected_preds = apply_postprocessor_to_split(df_eval, postproc, prot_attr=prot_attr)

    metrics = get_metrics(
        y_true=df_eval['label_binary'].values,
        y_pred=corrected_preds,
        prot_attr=df_eval[prot_attr].values,
        priv_group=1,
        pos_label=1,
    )
    metrics['method'] = label
    metrics['protected_attribute'] = prot_attr

    return metrics

print("✓ Fonction apply_post_processing définie.")

✓ Fonction apply_post_processing définie.


### 3.1 Reject Option Classification (ROC) sur RW Genre

In [98]:
# === ROC sur RW Genre (Genre puis Position) ===
print("Application de ROC sur RW Genre — attribut Genre...")
metrics_roc_rw_gender_gender = apply_post_processing(
    df_gender_val,
    df_gender,
    method='ROC',
    prot_attr='Gender_Binary',
    label='RW Genre + ROC',
)
print_metrics(metrics_roc_rw_gender_gender, "RW Genre + ROC — Fairness par Genre (valid)")

print("\nApplication de ROC sur RW Genre — attribut Position...")
metrics_roc_rw_gender_pos = apply_post_processing(
    df_gender_val,
    df_gender,
    method='ROC',
    prot_attr='Position_Binary',
    label='RW Genre + ROC',
)
print_metrics(metrics_roc_rw_gender_pos, "RW Genre + ROC — Fairness par Position (valid)")

Application de ROC sur RW Genre — attribut Genre...
  ROC — Seuil optimal : 0.5247
  ROC — Marge optimale : 0.0000

RW Genre + ROC — Fairness par Genre (valid)
  base_rate_truth                               : +0.4326
  statistical_parity_difference                 : +0.0428
  disparate_impact_ratio                        : 1.2106
  base_rate_preds                               : +0.2231
  equal_opportunity_difference                  : -0.0110
  average_odds_difference                       : +0.0252
  conditional_demographic_disparity             : +0.0045
  smoothed_edf                                  : +0.1907
  df_bias_amplification                         : +0.0541
  balanced_accuracy_score                       : +0.6107

Application de ROC sur RW Genre — attribut Position...
  ROC — Seuil optimal : 0.5742
  ROC — Marge optimale : 0.0000

RW Genre + ROC — Fairness par Position (valid)
  base_rate_truth                               : +0.4326
  statistical_parity_difference     

##### Analyse de la combinaison : RW Genre + ROC

1. La Balanced Accuracy (BA) monte à 0.6375 (optimisation sur le genre) et atteint même 0.6498 (optimisation sur la position), surpassant le modèle RW Genre seul (qui était à 0.6293). Cette amélioration s'explique par l'ajustement du seuil de classification, qui compense la tendance du modèle à sous-prédire (le base_rate_preds baisse autour de 0.26 - 0.32, se rapprochant de la réalité, ce qui réduit les Faux Positifs).

2. Impact sur l'attribut Genre (Cible : Genre)

Équité globale : L'optimisation ROC sur le genre dégrade paradoxalement un peu la parité statistique globale. Le DIR s'éloigne de 1 pour atteindre 1.2027 (contre 1.1515 avant post-processing), et le SPD remonte légèrement (+0.0488).

Équité d'opportunité : En revanche, l'EOD se rapproche fortement de 0 (+0.0047). Le modèle a parfaitement égalisé sa capacité à détecter les Vrais Positifs chez les hommes et les femmes.

3. Impact sur l'attribut Position (Cible : Position)

Le meilleur compromis observé : Appliquer ROC sur la position donne de très bons résultats avec une performance au maximum (BA de 0.6498), et le biais d'acquisition qui diminue.

Réduction du biais : Le DIR sur la position descend à 1.1193 (mieux que les 1.1429 de RW Genre seul) et we maintient dans la zone des 80% (0.8 - 1.25). L'amplification des biais est également contenue (+0.0584).

==> Conclusion

La combinaison RW Genre (Pre-processing) + ROC optimisé sur la Position (Post-processing) est pour l'instant la plus efficace.

### 3.2 Calibrated Equalized Odds sur RW Genre

In [99]:
# === Calibrated EqOdds sur RW Genre (Genre puis Position) ===
print("Application de Calibrated Equalized Odds sur RW Genre — attribut Genre...")
metrics_ceqo_rw_gender_gender = apply_post_processing(
    df_gender_val,
    df_gender,
    method='CalibratedEqOdds',
    prot_attr='Gender_Binary',
    label='RW Genre + CalEqOdds',
)
print_metrics(metrics_ceqo_rw_gender_gender, "RW Genre + Calibrated EqOdds — Fairness par Genre (valid)")

print("\nApplication de Calibrated Equalized Odds sur RW Genre — attribut Position...")
metrics_ceqo_rw_gender_pos = apply_post_processing(
    df_gender_val,
    df_gender,
    method='CalibratedEqOdds',
    prot_attr='Position_Binary',
    label='RW Genre + CalEqOdds',
)
print_metrics(metrics_ceqo_rw_gender_pos, "RW Genre + Calibrated EqOdds — Fairness par Position (valid)")

Application de Calibrated Equalized Odds sur RW Genre — attribut Genre...

RW Genre + Calibrated EqOdds — Fairness par Genre (valid)
  base_rate_truth                               : +0.4326
  statistical_parity_difference                 : +0.0632
  disparate_impact_ratio                        : 1.2709
  base_rate_preds                               : +0.2624
  equal_opportunity_difference                  : +0.0047
  average_odds_difference                       : +0.0422
  conditional_demographic_disparity             : +0.0060
  smoothed_edf                                  : +0.2392
  df_bias_amplification                         : +0.1026
  balanced_accuracy_score                       : +0.6382

Application de Calibrated Equalized Odds sur RW Genre — attribut Position...

RW Genre + Calibrated EqOdds — Fairness par Position (valid)
  base_rate_truth                               : +0.4326
  statistical_parity_difference                 : -0.2396
  disparate_impact_ratio        

##### Analyse de la combinaison : RW Genre + Calibrated Equalized Odds

1. **Application sur l'attribut Genre : Absence d'impact notable**
L'application de la méthode *Calibrated Equalized Odds* sur le genre ne modifie pas les résultats par rapport au *RW Genre* seul. La *Balanced Accuracy* (**0.6293**), le **DIR** (**1.1515**) et le **SPD** (**+0.0331**) restent identiques. Le pre-processing ayant déjà amené le modèle à un niveau de parité satisfaisant sur cet axe, l'algorithme de post-processing ne trouve pas d'ajustement supplémentaire pertinent sans dégrader l'erreur globale. L'effet se limite à une simple baisse du taux global de prédiction positive (`base_rate_preds` à 0.2335).

2. **Application sur l'attribut Position : Dégradation des prédictions**
Appliquée à l'attribut position, la méthode montre des limites claires. 
* La *Balanced Accuracy* baisse à **0.5720**.
* Le **DIR** tombe à **0.0000**, ce qui indique une asymétrie extrême : le modèle cesse presque entièrement de prédire la classe positive pour l'un des groupes. 
* Les écarts sur les erreurs conditionnelles augmentent fortement (l'**EOD** passe à **-0.3609** et l'amplification des biais à **+4.6032**). 
En cherchant à forcer l'égalité stricte des taux d'erreur sur une variable (la Position) fortement liée à l'état clinique du patient, la méthode écrase la capacité de prédiction du modèle (le taux de prédiction positive global chute à 13.7%).

==> Conclusion

La méthode *Calibrated Equalized Odds* n'est pas adaptée à notre configuration : elle est redondante sur l'attribut genre et dégrade significativement les performances sur l'attribut position.

### 3.3 Reject Option Classification et Calibrated Equalized Odds sur Baseline

Pour établir la **ligne de référence**, nous appliquons le post-processing sur Baseline (sans pré-processing). Cela nous permet de quantifier l'amélioration apportée par la combinaison RW Genre + post-processing par rapport à un modèle brut amélioré par post-processing uniquement.

In [100]:
# === ROC sur Baseline (Genre puis Position) ===
print("Application de ROC sur Baseline — attribut Genre...")
metrics_roc_baseline_gender = apply_post_processing(
    df_baseline_val,
    df_baseline,
    method='ROC',
    prot_attr='Gender_Binary',
    label='Baseline + ROC',
)
print_metrics(metrics_roc_baseline_gender, "Baseline + ROC — Fairness par Genre (valid)")

print("\nApplication de ROC sur Baseline — attribut Position...")
metrics_roc_baseline_pos = apply_post_processing(
    df_baseline_val,
    df_baseline,
    method='ROC',
    prot_attr='Position_Binary',
    label='Baseline + ROC',
)
print_metrics(metrics_roc_baseline_pos, "Baseline + ROC — Fairness par Position (valid)")

# === Calibrated EqOdds sur Baseline (Genre puis Position) ===
print("\n\nApplication de Calibrated Equalized Odds sur Baseline — attribut Genre...")
metrics_ceqo_baseline_gender = apply_post_processing(
    df_baseline_val,
    df_baseline,
    method='CalibratedEqOdds',
    prot_attr='Gender_Binary',
    label='Baseline + CalEqOdds',
)
print_metrics(metrics_ceqo_baseline_gender, "Baseline + Calibrated EqOdds — Fairness par Genre (valid)")

print("\nApplication de Calibrated Equalized Odds sur Baseline — attribut Position...")
metrics_ceqo_baseline_pos = apply_post_processing(
    df_baseline_val,
    df_baseline,
    method='CalibratedEqOdds',
    prot_attr='Position_Binary',
    label='Baseline + CalEqOdds',
)
print_metrics(metrics_ceqo_baseline_pos, "Baseline + Calibrated EqOdds — Fairness par Position (valid)")

Application de ROC sur Baseline — attribut Genre...
  ROC — Seuil optimal : 0.4852
  ROC — Marge optimale : 0.0000

Baseline + ROC — Fairness par Genre (valid)
  base_rate_truth                               : +0.4326
  statistical_parity_difference                 : +0.0420
  disparate_impact_ratio                        : 1.1129
  base_rate_preds                               : +0.3917
  equal_opportunity_difference                  : -0.0380
  average_odds_difference                       : +0.0150
  conditional_demographic_disparity             : +0.0032
  smoothed_edf                                  : +0.1068
  df_bias_amplification                         : -0.0298
  balanced_accuracy_score                       : +0.6715

Application de ROC sur Baseline — attribut Position...
  ROC — Seuil optimal : 0.2080
  ROC — Marge optimale : 0.0000

Baseline + ROC — Fairness par Position (valid)
  base_rate_truth                               : +0.4326
  statistical_parity_difference     

##### Analyse du Post-processing sur la Baseline

1. **Application de Reject Option Classification (ROC)**
Comme observé précédemment, la marge optimale trouvée est de 0.0000, ce qui indique que l'algorithme a agi uniquement comme un optimiseur de seuil global (décalage à 0.4753 pour le genre et 0.5247 pour la position) sans exploiter de zone d'incertitude.

* **Sur l'attribut Genre :** La performance globale est maintenue avec une très légère hausse (la *Balanced Accuracy* passe de 0.6174 à **0.6191**). Les métriques d'équité restent stables et acceptables : le **DIR** s'établit à **1.1726** (contre 1.1525 initialement) et l'**EOD** est pratiquement nul (**-0.0028**), indiquant une très bonne égalité d'opportunité entre les genres.
* **Sur l'attribut Position :** La performance subit une baisse très marginale (BA de **0.6155**). En revanche, la correction du biais matériel est efficace : le **DIR** descend à **1.1862** (contre 1.2388 pour la baseline), s'éloignant de la limite de tolérance. Le **SPD** est également réduit (+0.0417). 

2. **Application de Calibrated Equalized Odds**
Cette méthode produit des résultats beaucoup plus diverses sur les prédictions brutes de la baseline.

* **Sur l'attribut Genre :** L'application de l'algorithme entraîne une détérioration sévère du modèle. La *Balanced Accuracy* chute à **0.5801**. Plus problématique, l'équité est détruite : le **DIR** atteint une valeur extrême de **3.8262**, et l'**EOD** grimpe à **+0.2701**. En tentant de forcer la calibration mathématique sur les prédictions brutes, la méthode a créé un déséquilibre majeur qui n'existait pas.
* **Sur l'attribut Position :** L'algorithme n'a eu aucun effet. Les métriques renvoyées (BA de **0.6174**, DIR de **1.2388**, SPD de **+0.0582**) sont strictement identiques à celles de la baseline avant post-processing. L'algorithme n'a trouvé aucun ajustement réalisable pour égaliser les cotes (Odds) sans dégrader la fonction d'erreur, et a donc conservé les prédictions initiales.

##### Conclusion

L'évaluation du post-processing sur le modèle non pondéré (Baseline) confirme les dynamiques observées précédemment. La méthode **Calibrated Equalized Odds** se révèle inadaptée (soit destructrice, soit inactive). À l'inverse, **ROC** démontre une bonne stabilité : il parvient à réduire le biais lié à la position de vue (ramenant le DIR à 1.18) sans compromettre l'utilité clinique du modèle de base.

### 3.4 Post-processing sur RW Position (comparaison avec RW Genre)

Pour comparer, nous testons le post-processing sur RW Position, la méthode de pré-processing la moins bonne du point de vue performance/fairness. On veut ici montrer si le post-processing peut tout de même améliorer une stratégie pré-processing sous-optimale.

In [101]:
# === Combinaisons pre+post sur RW Genre et RW Position ===
# Note: les résultats RW Genre + ROC et RW Genre + CalEqOdds ont déjà été calculés en 3.2 et 3.3
# On teste ici RW Position avec post-processing, pour comparaison avec RW Genre

required = ['df_gender_val', 'df_gender', 'df_pos_val', 'df_pos']
missing = [v for v in required if v not in globals()]
if missing:
    raise RuntimeError(
        f"Variables manquantes pour les combinaisons pre+post: {missing}. "
        "Exécuter d'abord les cellules d'entraînement et de prédiction précédentes."
    )

print("APPLICATION DE POST-PROCESSING SUR RW POSITION (pour comparaison)")
print("=" * 70)

print("\nApplication de ROC sur RW Position — attribut Genre...")
metrics_roc_rw_pos_gender = apply_post_processing(
    df_pos_val,
    df_pos,
    method='ROC',
    prot_attr='Gender_Binary',
    label='RW Position + ROC',
)
print_metrics(metrics_roc_rw_pos_gender, "RW Position + ROC — Fairness par Genre (valid)")

print("\nApplication de ROC sur RW Position — attribut Position...")
metrics_roc_rw_pos_pos = apply_post_processing(
    df_pos_val,
    df_pos,
    method='ROC',
    prot_attr='Position_Binary',
    label='RW Position + ROC',
)
print_metrics(metrics_roc_rw_pos_pos, "RW Position + ROC — Fairness par Position (valid)")

print("\nApplication de CalEqOdds sur RW Position — attribut Genre...")
metrics_ceqo_rw_pos_gender = apply_post_processing(
    df_pos_val,
    df_pos,
    method='CalibratedEqOdds',
    prot_attr='Gender_Binary',
    label='RW Position + CalEqOdds',
)
print_metrics(metrics_ceqo_rw_pos_gender, "RW Position + CalEqOdds — Fairness par Genre (valid)")

print("\nApplication de CalEqOdds sur RW Position — attribut Position...")
metrics_ceqo_rw_pos_pos = apply_post_processing(
    df_pos_val,
    df_pos,
    method='CalibratedEqOdds',
    prot_attr='Position_Binary',
    label='RW Position + CalEqOdds',
)
print_metrics(metrics_ceqo_rw_pos_pos, "RW Position + CalEqOdds — Fairness par Position (valid)")

APPLICATION DE POST-PROCESSING SUR RW POSITION (pour comparaison)

Application de ROC sur RW Position — attribut Genre...
  ROC — Seuil optimal : 0.4654
  ROC — Marge optimale : 0.0000

RW Position + ROC — Fairness par Genre (valid)
  base_rate_truth                               : +0.4326
  statistical_parity_difference                 : -0.0267
  disparate_impact_ratio                        : 0.9550
  base_rate_preds                               : +0.5811
  equal_opportunity_difference                  : -0.0772
  average_odds_difference                       : -0.0466
  conditional_demographic_disparity             : -0.0020
  smoothed_edf                                  : +0.0635
  df_bias_amplification                         : -0.0731
  balanced_accuracy_score                       : +0.6337

Application de ROC sur RW Position — attribut Position...
  ROC — Seuil optimal : 0.5346
  ROC — Marge optimale : 0.0855

RW Position + ROC — Fairness par Position (valid)
  base_rate_tru

##### Conclusion : Le post-processing peut-il améliorer un mauvais pre-processing ?

L'algorithme **ROC (optimisé sur la position)** relève la balanced_accuracy de *RW Position* passant de de 0.55 à **0.6535**, et le modèle corrige sa sur-correction (le DIR remonte à **1.1235**). À l'inverse, *Calibrated EqOdds* échoue de nouveau à maintenir la performance clinique (BA < 0.57).



### 3.5 Comparaison globale et discussion des compromis

In [102]:
# === Tableau récapitulatif global ===
required = [
    'metrics_baseline_gender', 'metrics_baseline_pos',
    'metrics_rw_gender', 'metrics_rw_pos', 'metrics_rw_pos_gender',
    'metrics_rw_cross_gender', 'metrics_rw_cross_pos',
    'metrics_roc_baseline_gender', 'metrics_roc_baseline_pos',
    'metrics_ceqo_baseline_gender', 'metrics_ceqo_baseline_pos',
    'metrics_roc_rw_gender_gender', 'metrics_roc_rw_gender_pos',
    'metrics_ceqo_rw_gender_gender', 'metrics_ceqo_rw_gender_pos',
    'metrics_roc_rw_pos_gender', 'metrics_roc_rw_pos_pos',
    'metrics_ceqo_rw_pos_gender', 'metrics_ceqo_rw_pos_pos',
]
missing = [v for v in required if v not in globals()]
if missing:
    raise RuntimeError(
        f"Variables manquantes pour le tableau global: {missing}. "
        "Exécuter d'abord toutes les cellules d'entraînement, de post-processing et de combinaisons."
    )

all_metrics = [
    {**metrics_baseline_gender, 'method': 'Baseline', 'attr': 'Genre'},
    {**metrics_baseline_pos, 'method': 'Baseline', 'attr': 'Position'},
    {**metrics_rw_gender, 'method': 'RW Genre', 'attr': 'Genre'},
    {**metrics_rw_pos_gender, 'method': 'RW Position', 'attr': 'Genre'},
    {**metrics_rw_pos, 'method': 'RW Position', 'attr': 'Position'},
    {**metrics_rw_cross_gender, 'method': 'RW Genre×Position', 'attr': 'Genre'},
    {**metrics_rw_cross_pos, 'method': 'RW Genre×Position', 'attr': 'Position'},
    {**metrics_roc_baseline_gender, 'method': 'Baseline + ROC', 'attr': 'Genre'},
    {**metrics_roc_baseline_pos, 'method': 'Baseline + ROC', 'attr': 'Position'},
    {**metrics_ceqo_baseline_gender, 'method': 'Baseline + CalEqOdds', 'attr': 'Genre'},
    {**metrics_ceqo_baseline_pos, 'method': 'Baseline + CalEqOdds', 'attr': 'Position'},
    {**metrics_roc_rw_gender_gender, 'method': 'RW Genre + ROC', 'attr': 'Genre'},
    {**metrics_roc_rw_gender_pos, 'method': 'RW Genre + ROC', 'attr': 'Position'},
    {**metrics_ceqo_rw_gender_gender, 'method': 'RW Genre + CalEqOdds', 'attr': 'Genre'},
    {**metrics_ceqo_rw_gender_pos, 'method': 'RW Genre + CalEqOdds', 'attr': 'Position'},
    {**metrics_roc_rw_pos_gender, 'method': 'RW Position + ROC', 'attr': 'Genre'},
    {**metrics_roc_rw_pos_pos, 'method': 'RW Position + ROC', 'attr': 'Position'},
    {**metrics_ceqo_rw_pos_gender, 'method': 'RW Position + CalEqOdds', 'attr': 'Genre'},
    {**metrics_ceqo_rw_pos_pos, 'method': 'RW Position + CalEqOdds', 'attr': 'Position'},
]

df_compare = pd.DataFrame(all_metrics)
cols_display = [
    'method', 'attr', 'balanced_accuracy_score',
    'statistical_parity_difference', 'disparate_impact_ratio',
    'equal_opportunity_difference', 'average_odds_difference',
    'df_bias_amplification'
]

# Mets show_table=True si tu veux afficher le tableau complet dans le notebook.
show_table = False
if show_table:
    print("=" * 100)
    print("TABLEAU RÉCAPITULATIF — Toutes les méthodes et les deux attributs sensibles")
    print("=" * 100)
    display(df_compare[cols_display].sort_values(['attr', 'method']).round(4))
else:
    print("df_compare créé (affichage désactivé).")

df_compare créé (affichage désactivé).


In [103]:
# === Synthèse triée pour lecture rapide ===
if 'df_compare' not in globals():
    raise RuntimeError("df_compare absent. Exécuter d'abord la cellule 40 (tableau global).")

for attr in ['Genre', 'Position']:
    print(f"\n=== Top méthodes pour {attr} ===")
    df_attr = df_compare[df_compare['attr'] == attr].copy()
    df_attr['score_tradeoff'] = (
        df_attr['balanced_accuracy_score']
        - 0.5 * df_attr['statistical_parity_difference'].abs()
        - 0.5 * (df_attr['disparate_impact_ratio'] - 1.0).abs()
    )
    keep = ['method', 'balanced_accuracy_score', 'statistical_parity_difference', 'disparate_impact_ratio', 'score_tradeoff']
    display(df_attr[keep].sort_values('score_tradeoff', ascending=False).head(5).round(4))


=== Top méthodes pour Genre ===


,method,balanced_accuracy_score,statistical_parity_difference,disparate_impact_ratio,score_tradeoff
15,RW Position + ROC,0.6337,-0.0267,0.9550,0.5979
0,Baseline,0.6722,0.0399,1.1147,0.5949
7,Baseline + ROC,0.6715,0.0420,1.1129,0.5940
3,RW Position,0.6216,-0.0213,0.9621,0.5920
11,RW Genre + ROC,0.6107,0.0428,1.2106,0.4840



=== Top méthodes pour Position ===


,method,balanced_accuracy_score,statistical_parity_difference,disparate_impact_ratio,score_tradeoff
16,RW Position + ROC,0.6424,0.0350,1.0727,0.5886
6,RW Genre×Position,0.6414,-0.0720,0.8773,0.5440
8,Baseline + ROC,0.5681,0.0434,1.0542,0.5193
4,RW Position,0.6216,-0.1182,0.8020,0.4635
12,RW Genre + ROC,0.5902,0.0435,1.2972,0.4199


- Sur Genre: RW Genre reste la référence propre, et RW Genre + CalEqOdds est identique ici (donc pas de gain réel).
- Sur Position: RW Position + ROC et RW Genre + ROC passent en tête car ROC améliore fortement la performance sans exploser les métriques fairness.
- CalEqOdds reste globalement peu utile / instable ici : ROC est le post-processing le plus utile.
- En privilégiant la robustesse globale: RW Genre (+ éventuellement ROC) reste le meilleur fil conducteur.

## 4/- Conclusion

### 4.1 Rappel de l'objectif

L'objectif de ce projet était d'identifier une stratégie de mitigation des biais offrant le meilleur compromis entre **performance** (Balanced Accuracy) et **équité** (SPD, DIR, EOD, AOD), sur deux attributs sensibles : **le genre** et **la position de vue (PA/AP)**.

Nous avons évalué successivement :
- une baseline sans pondération,
- des stratégies de pre-processing (RW Genre, RW Position, RW Genre×Position),
- des combinaisons pre + post-processing (ROC, Calibrated Equalized Odds).

### 4.2 Synthèse des résultats par attribut

**Attribut Genre**
- La baseline était déjà relativement correcte sur le genre.
- Le pre-processing **RW Genre** améliore légèrement le compromis global sans coût majeur.
- Le post-processing **ROC** sur RW Genre améliore la performance (BA) mais avec un gain fairness plus limité.
- **Calibrated EqOdds** n'apporte pas de bénéfice robuste dans notre configuration.

**Attribut Position**
- C'est l'attribut le plus difficile : la baseline est proche d'une zone de biais technique.
- **RW Position** seul sur-corrige et dégrade fortement la performance.
- L'ajout de **ROC** corrige en partie cette faiblesse et donne la meilleure performance sur Position.
- Toutefois, cette option est moins stable lorsqu'on regarde l'ensemble des objectifs (deux attributs + robustesse globale).

### 4.3 Modèle gagnant et justification

Au vu de l'ensemble des résultats, le modèle retenu comme **gagnant global** est :

**RW Genre + ROC**

- Il maintient un niveau de performance élevé sur les deux lectures (Genre et Position), avec une BA élevée (environ 0.64/0.65 selon l'attribut d'évaluation).
- Il évite les sur-corrections observées avec RW Position seul ou RW Genre×Position.
- Il reste compétitif sur les métriques de fairness sans effondrement sur un attribut.
- Dans les classements Position, **RW Position + ROC** peut être légèrement meilleur en score composite, mais l'écart est faible et moins robuste quand on réintègre la lecture multi-attributs.

### 4.4 Limites

- Une **Balanced Accuracy autour de 64%** reste insuffisante pour envisager un usage clinique réel : le modèle conserve un niveau d'erreur important.
- Les expériences ont été réalisées en **mode smoke test (1 époque)** pour des raisons de temps et de ressources, ce qui limite la stabilité des conclusions.
- Les résultats peuvent varier selon l'échantillonnage, il serait intéressant d'observer le comportement du modèle sur d'autres données.
- Nous avons choisi de centrer nos analyses autour de **deux attributs sensibles** (genre, position), mais d'autres dimensions comme l'age ou la comorbidité auraient pu être intéressantes à étudier.

### 4.5 Perspectives - la suite du projet, comment le poursuivre/l'améliorer

- Passer du smoke test à un entraînement complet (plus d'époques, réglage d'hyperparamètres, arrêt anticipé) pour mesurer le vrai potentiel de performance.
- Tester des méthodes de mitigation complémentaires (in-processing, contraintes fairness pendant l'entraînement) et comparer leur stabilité à ROC.
- Compléter l'évaluation par des métriques cliniques opérationnelles (sensibilité/spécificité par groupe, calibration, courbes PR/ROC) afin de relier fairness et utilité médicale.
- Évaluer la généralisabilité sur un jeu externe (autre hôpital, autre protocole d'acquisition) avant toute conclusion sur un déploiement possible.